# Generate the Final Balanced Execution-Time Dataset (25K)

This notebook generates the dataset only. It does **not** train a GNN.

The input is the same set of 600 final DAGs stored under:

```text
~/dag_dataset/
```

To preserve the split ratio used in the project design, each DAG produces exactly 42 unique execution contexts:

\[
6\ \text{target nodes} \times 7\ \text{contexts} = 42
\]

This gives:

- train: 16,800 records
- validation: 2,100 records
- calibration: 2,100 records
- test-ID: 2,100 records
- test-OOD: 2,100 records
- total: 25,200 records

The split ratio therefore remains:

\[
8:1:1:1:1
\]

For the dynamic system state \(z_t\), the notebook does not create several isolated discrete distributions. Instead, it samples a continuous latent system-load variable from one shared Beta distribution and generates correlated state variables from it. This provides coverage from light to heavy system load while keeping the overall data-generating process continuous and highly overlapping across splits.

The new dataset is written to:

```text
~/dag_runtime_dataset_25k/
```

The target `y_exec_us` is generated from the execution-time simulator based on project equations (39) and (40).

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
import shutil
from collections import Counter
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

SOURCE_ROOT = Path.home() / "dag_dataset"
SOURCE_MANIFEST = SOURCE_ROOT / "manifests" / "all_graphs.csv"
DEST_ROOT = Path.home() / "dag_runtime_dataset_25k"

SPLITS = ["train", "validation", "calibration", "test_id", "test_ood"]
EXPECTED_GRAPH_COUNTS = {
    "train": 400,
    "validation": 50,
    "calibration": 50,
    "test_id": 50,
    "test_ood": 50,
}
RECORDS_PER_GRAPH = 42
EXPECTED_RECORD_COUNTS = {
    "train": EXPECTED_GRAPH_COUNTS["train"] * RECORDS_PER_GRAPH,
    "validation": EXPECTED_GRAPH_COUNTS["validation"] * RECORDS_PER_GRAPH,
    "calibration": EXPECTED_GRAPH_COUNTS["calibration"] * RECORDS_PER_GRAPH,
    "test_id": EXPECTED_GRAPH_COUNTS["test_id"] * RECORDS_PER_GRAPH,
    "test_ood": EXPECTED_GRAPH_COUNTS["test_ood"] * RECORDS_PER_GRAPH,
}
BASE_SEED = 20260807
OVERWRITE_DESTINATION = False

assert SOURCE_ROOT.exists(), f"Source dataset not found: {SOURCE_ROOT}"
assert SOURCE_MANIFEST.exists(), f"Manifest not found: {SOURCE_MANIFEST}"

print("Source:", SOURCE_ROOT)
print("Destination:", DEST_ROOT)

## 1) Hardware and Simulator Constants

The DVFS frequencies and voltages are taken directly from project equations (11) and (12).

The project specification does not assign exact numeric values to \(IPS_{c,\ell}\), \(B^{max}_{r(c),\ell}\), the \(\eta\) coefficients, interference coefficients, or the numeric ranges of synthetic features. Therefore, the corresponding values below are explicit simulator implementation choices. They are saved in `generation_config.json` for reproducibility and documentation.

In [ ]:
DVFS_TABLE = {
    "big": [
        {"level": 0, "frequency_ghz": 1.0, "voltage_v": 0.90, "bmax_gbps": 8.0},
        {"level": 1, "frequency_ghz": 1.4, "voltage_v": 0.98, "bmax_gbps": 10.0},
        {"level": 2, "frequency_ghz": 1.8, "voltage_v": 1.06, "bmax_gbps": 12.0},
        {"level": 3, "frequency_ghz": 2.2, "voltage_v": 1.15, "bmax_gbps": 14.0},
    ],
    "little": [
        {"level": 0, "frequency_ghz": 0.6, "voltage_v": 0.80, "bmax_gbps": 5.0},
        {"level": 1, "frequency_ghz": 0.9, "voltage_v": 0.86, "bmax_gbps": 6.5},
        {"level": 2, "frequency_ghz": 1.2, "voltage_v": 0.93, "bmax_gbps": 8.0},
        {"level": 3, "frequency_ghz": 1.5, "voltage_v": 1.00, "bmax_gbps": 9.5},
    ],
}

CORE_CAPACITY_FACTOR = {"big": 1.00, "little": 0.65}
CORES = {"big": ["big_0", "big_1"], "little": ["little_0", "little_1"]}

ETA_1 = 0.12
ETA_2 = 0.50
ETA_3 = 0.25
GAMMA = 1.0
CACHE_INTERFERENCE_COEF = 0.08
MEM_INTERFERENCE_COEF = 0.10
OS_INTERFERENCE_COEF = 0.05
OS_FIXED_OVERHEAD_US = 3.0
NOISE_STD_RATIO = 0.03
REFERENCE_IPS = 1.8e9
REFERENCE_BW = 12.0e9

hardware_rows = []
for core_type, levels in DVFS_TABLE.items():
    for level_cfg in levels:
        frequency_hz = level_cfg["frequency_ghz"] * 1e9
        effective_ips = frequency_hz * CORE_CAPACITY_FACTOR[core_type]
        hardware_rows.append({
            "core_type": core_type,
            "dvfs_level": level_cfg["level"],
            "frequency_ghz": level_cfg["frequency_ghz"],
            "voltage_v": level_cfg["voltage_v"],
            "effective_ips_per_sec": effective_ips,
            "nominal_bandwidth_bytes_per_sec": level_cfg["bmax_gbps"] * 1e9,
        })

hardware_df = pd.DataFrame(hardware_rows)
display(hardware_df)

## 2) Node-Feature Distributions: Diverse but Overlapping

Three operation types are used:

- `compute_bound`
- `balanced`
- `memory_bound`

The Gaussian distributions for \(C_v\) and \(M_v\) intentionally overlap substantially. This avoids creating three artificial, clearly separated clusters while still preserving meaningful differences between operation types.

The parameter \(\lambda_v\) is used only by the simulator and is **not** a GNN input feature.

In [ ]:

NODE_PROFILES = {
    "compute_bound": {
        "probability": 1/3,
        "compute_cycles_mean": 95e6,
        "compute_cycles_std": 28e6,
        "compute_cycles_min": 25e6,
        "compute_cycles_max": 180e6,
        "memory_bytes_mean": 28 * 1024**2,
        "memory_bytes_std": 14 * 1024**2,
        "memory_bytes_min": 4 * 1024**2,
        "memory_bytes_max": 80 * 1024**2,
        "lambda_v": 0.55,
    },
    "balanced": {
        "probability": 1/3,
        "compute_cycles_mean": 80e6,
        "compute_cycles_std": 28e6,
        "compute_cycles_min": 20e6,
        "compute_cycles_max": 170e6,
        "memory_bytes_mean": 40 * 1024**2,
        "memory_bytes_std": 18 * 1024**2,
        "memory_bytes_min": 8 * 1024**2,
        "memory_bytes_max": 96 * 1024**2,
        "lambda_v": 0.80,
    },
    "memory_bound": {
        "probability": 1/3,
        "compute_cycles_mean": 65e6,
        "compute_cycles_std": 25e6,
        "compute_cycles_min": 15e6,
        "compute_cycles_max": 150e6,
        "memory_bytes_mean": 54 * 1024**2,
        "memory_bytes_std": 22 * 1024**2,
        "memory_bytes_min": 12 * 1024**2,
        "memory_bytes_max": 128 * 1024**2,
        "lambda_v": 1.05,
    },
}

display(pd.DataFrame(NODE_PROFILES).T)


## 3) Continuous and Correlated System State \(z_t\)

The system-state vector is:

\[
z_t=[
u^{cpu},
q^{ready},
n^{active},
n^{mem},
u^{bus},
\Delta T,
J^{release}
]
\]

A latent continuous system-load variable is first sampled as:

\[
L\sim Beta(2.2,2.2)
\]

The components of \(z_t\) are then generated as correlated noisy functions of \(L\). This creates a smooth continuum of operating conditions rather than separate `normal`, `loaded`, and `stressed` islands.

`generator_load_score` is generator metadata only. It is **not** provided to the model.

In [ ]:

LOAD_BETA_A = 2.2
LOAD_BETA_B = 2.2

SYSTEM_STATE_CONFIG = {
    "load_beta_a": LOAD_BETA_A,
    "load_beta_b": LOAD_BETA_B,
    "cpu_noise_std": 0.055,
    "bus_noise_std": 0.060,
    "thermal_noise_std": 0.055,
    "active_core_noise_std": 0.45,
    "memory_task_noise_std": 0.55,
}

display(pd.DataFrame([SYSTEM_STATE_CONFIG]))


## 4) DOT Parsing and Topological Features

Only two explicit global topological features are added:

- `topo_level`
- `reverse_topo_level`

The graph adjacency still provides local topology directly to the GNN. These two scalar features summarize the node's global position inside the DAG, which can be useful for a shallow message-passing network.

In [ ]:
EDGE_RE = re.compile(r'^\s*"?([A-Za-z0-9_.:-]+)"?\s*->\s*"?([A-Za-z0-9_.:-]+)"?')
NODE_RE = re.compile(r'^\s*"?([A-Za-z0-9_.:-]+)"?\s*\[')
RESERVED_DOT_WORDS = {"graph", "digraph", "strict", "node", "edge", "subgraph"}


def numeric_sort_key(value: str):
    text = str(value)
    return (0, int(text)) if text.isdigit() else (1, text)


def stable_seed(*parts: object) -> int:
    text = ":".join(str(part) for part in (BASE_SEED, *parts))
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little", signed=False)


def parse_daggen_dot(dot_path: Path) -> nx.DiGraph:
    graph = nx.DiGraph()
    with dot_path.open("r", encoding="utf-8", errors="replace") as file:
        for line in file:
            edge_match = EDGE_RE.match(line)
            if edge_match:
                source, target = edge_match.groups()
                graph.add_edge(str(source), str(target))
                continue
            node_match = NODE_RE.match(line)
            if node_match:
                node_id = node_match.group(1)
                if node_id.lower() not in RESERVED_DOT_WORDS:
                    graph.add_node(str(node_id))
    if graph.number_of_nodes() == 0:
        raise ValueError(f"No nodes parsed from {dot_path}")
    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError(f"Not a DAG: {dot_path}")
    return graph


def compute_levels(graph: nx.DiGraph):
    topo_order = list(nx.topological_sort(graph))
    topo_level = {}
    for node in topo_order:
        preds = list(graph.predecessors(node))
        topo_level[node] = 0 if not preds else 1 + max(topo_level[p] for p in preds)
    reverse_level = {}
    for node in reversed(topo_order):
        succs = list(graph.successors(node))
        reverse_level[node] = 0 if not succs else 1 + max(reverse_level[s] for s in succs)
    return topo_order, topo_level, reverse_level


def clipped_gaussian(rng, mean, std, minimum, maximum) -> int:
    return max(1, int(round(float(np.clip(rng.normal(mean, std), minimum, maximum)))))

## 5) Static Node and Edge Features

The files under `features/nodes/...` contain only the node attributes intended for the GNN input vector:

```text
compute_cycles
memory_bytes
input_size_bytes
in_degree
out_degree
operation_type
criticality
topo_level
reverse_topo_level
```

The simulator-only parameter \(\lambda_v\) is stored separately under `simulator/node_params/...` so that it is not accidentally used as a model input.

The only learned edge attribute is:

```text
data_bytes = d_uv
```

`source` and `target` define graph topology and are not learned edge features.

In [ ]:
def assign_node_workloads(graph: nx.DiGraph, rng: np.random.Generator):
    names = list(NODE_PROFILES)
    probs = np.array([NODE_PROFILES[name]["probability"] for name in names], dtype=float)
    probs = probs / probs.sum()
    result = {}
    for node in sorted(graph.nodes(), key=numeric_sort_key):
        operation_type = str(rng.choice(names, p=probs))
        cfg = NODE_PROFILES[operation_type]
        result[node] = {
            "compute_cycles": clipped_gaussian(rng, cfg["compute_cycles_mean"], cfg["compute_cycles_std"], cfg["compute_cycles_min"], cfg["compute_cycles_max"]),
            "memory_bytes": clipped_gaussian(rng, cfg["memory_bytes_mean"], cfg["memory_bytes_std"], cfg["memory_bytes_min"], cfg["memory_bytes_max"]),
            "operation_type": operation_type,
            "lambda_v": float(cfg["lambda_v"]),
        }
    return result


def assign_edge_data_for_ccr(graph, node_workloads, ccr_target, rng):
    edges = list(graph.edges())
    if not edges:
        return {}
    total_compute_seconds_ref = sum(v["compute_cycles"] for v in node_workloads.values()) / REFERENCE_IPS
    target_total_comm_bytes = max(0.01, float(ccr_target)) * total_compute_seconds_ref * REFERENCE_BW
    weights = np.abs(rng.normal(loc=1.0, scale=0.50, size=len(edges)))
    weights = np.maximum(weights, 0.05)
    weights = weights / weights.sum()
    edge_data = {}
    for edge, weight in zip(edges, weights):
        raw_bytes = target_total_comm_bytes * float(weight)
        edge_data[edge] = max(4096, int(round(raw_bytes / 4096.0) * 4096))
    return edge_data


def longest_weighted_path_seconds(graph, topo_order, node_seconds):
    longest_finish = {}
    for node in topo_order:
        preds = list(graph.predecessors(node))
        if not preds:
            longest_finish[node] = node_seconds[node]
        else:
            best_pred = max(preds, key=lambda p: longest_finish[p])
            longest_finish[node] = longest_finish[best_pred] + node_seconds[node]
    end_node = max(topo_order, key=lambda n: longest_finish[n])
    return float(longest_finish[end_node])

def sample_ccr_target(rng: np.random.Generator) -> float:
    # Controlled CCR: same clipped log-normal family for all splits.
    value = float(np.exp(rng.normal(loc=0.0, scale=0.30)))
    return float(np.clip(value, 0.50, 2.00))


## 6) Task Metadata: \(T_i\), \(D_i\), and \(\chi_i\)

According to project equation (2), each task has task-level timing and criticality metadata. This notebook generates:

- `criticality`
- `deadline_us`
- `period_us`

For controlled calibration coverage, HI/LO criticality is distributed approximately 50/50 within each split.

The constraint \(D_i\le T_i\) is enforced. `deadline_us` and `period_us` are task metadata and are **not** inputs to the execution-time predictor. Criticality is also copied into the node table because the project allows it as part of the node feature vector.

In [ ]:
manifest = pd.read_csv(SOURCE_MANIFEST)
required_manifest_columns = {"graph_id", "split", "file_path"}
missing = required_manifest_columns - set(manifest.columns)
assert not missing, f"Missing manifest columns: {sorted(missing)}"
manifest = manifest.copy()
manifest["graph_id"] = manifest["graph_id"].astype(str)
actual_counts = manifest.groupby("split").size().to_dict()
assert actual_counts == EXPECTED_GRAPH_COUNTS, actual_counts

criticality_map = {}
for split in SPLITS:
    graph_ids = sorted(manifest.loc[manifest["split"] == split, "graph_id"].tolist())
    for index, graph_id in enumerate(graph_ids):
        criticality_map[graph_id] = "HI" if index % 2 == 0 else "LO"

display(manifest.groupby("split").size().rename("graph_count"))

In [ ]:
def get_manifest_float(row: pd.Series, column: str, default: float) -> float:
    if column not in row.index or pd.isna(row[column]):
        return default
    try:
        return float(row[column])
    except (TypeError, ValueError):
        return default


def build_graph_static_data(row: pd.Series):
    graph_id = str(row["graph_id"])
    split = str(row["split"])
    graph_path = SOURCE_ROOT / str(row["file_path"])
    rng = np.random.default_rng(stable_seed("static", graph_id))
    graph = parse_daggen_dot(graph_path)
    topo_order, topo_level, reverse_level = compute_levels(graph)
    workloads = assign_node_workloads(graph, rng)
    ccr_target = sample_ccr_target(rng)
    edge_data = assign_edge_data_for_ccr(graph, workloads, ccr_target, rng)

    input_size_bytes = {node: 0 for node in graph.nodes()}
    for (source, target), data_bytes in edge_data.items():
        input_size_bytes[target] += int(data_bytes)

    criticality = criticality_map[graph_id]
    node_rows, simulator_rows = [], []
    for node in sorted(graph.nodes(), key=numeric_sort_key):
        workload = workloads[node]
        node_rows.append({
            "graph_id": graph_id,
            "split": split,
            "node_id": node,
            "compute_cycles": int(workload["compute_cycles"]),
            "memory_bytes": int(workload["memory_bytes"]),
            "input_size_bytes": int(input_size_bytes[node]),
            "in_degree": int(graph.in_degree(node)),
            "out_degree": int(graph.out_degree(node)),
            "operation_type": workload["operation_type"],
            "criticality": criticality,
            "topo_level": int(topo_level[node]),
            "reverse_topo_level": int(reverse_level[node]),
        })
        simulator_rows.append({
            "graph_id": graph_id,
            "split": split,
            "node_id": node,
            "lambda_v": float(workload["lambda_v"]),
        })

    edge_rows = []
    for source, target in sorted(graph.edges(), key=lambda e: (numeric_sort_key(e[0]), numeric_sort_key(e[1]))):
        edge_rows.append({
            "graph_id": graph_id,
            "split": split,
            "source": source,
            "target": target,
            "data_bytes": int(edge_data[(source, target)]),
        })

    baseline_node_seconds = {}
    for node in graph.nodes():
        workload = workloads[node]
        baseline_node_seconds[node] = (
            workload["compute_cycles"] / REFERENCE_IPS
            + workload["lambda_v"] * workload["memory_bytes"] / REFERENCE_BW
            + GAMMA * input_size_bytes[node] / REFERENCE_BW
        )
    baseline_cp_seconds = longest_weighted_path_seconds(graph, topo_order, baseline_node_seconds)
    deadline_us = int(math.ceil(baseline_cp_seconds * float(rng.uniform(1.30, 1.80)) * 1e6))
    period_us = max(deadline_us, int(math.ceil(deadline_us * float(rng.uniform(1.00, 1.50)))))

    num_nodes, num_edges = graph.number_of_nodes(), graph.number_of_edges()
    max_possible_dag_edges = num_nodes * (num_nodes - 1) / 2
    level_counts = Counter(topo_level.values())
    total_compute_ref = sum(v["compute_cycles"] for v in workloads.values()) / REFERENCE_IPS
    total_comm_ref = sum(edge_data.values()) / REFERENCE_BW if edge_data else 0.0
    ccr_actual = total_comm_ref / total_compute_ref if total_compute_ref > 0 else 0.0

    task_row = {
        "graph_id": graph_id,
        "split": split,
        "criticality": criticality,
        "deadline_us": deadline_us,
        "period_us": period_us,
    }
    summary_row = {
        "graph_id": graph_id,
        "split": split,
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "depth": max(topo_level.values()) + 1,
        "max_width": max(level_counts.values()),
        "actual_density": num_edges / max_possible_dag_edges if max_possible_dag_edges > 0 else 0.0,
        "ccr_target": ccr_target,
        "ccr_actual": ccr_actual,
        "baseline_critical_path_us": baseline_cp_seconds * 1e6,
        "fat": row.get("fat", np.nan),
        "density_parameter": row.get("density", np.nan),
        "regular": row.get("regular", np.nan),
        "jump": row.get("jump", np.nan),
    }
    return graph, topo_order, topo_level, pd.DataFrame(node_rows), pd.DataFrame(edge_rows), pd.DataFrame(simulator_rows), task_row, summary_row

## 7) Select 6 Diverse Target Nodes per DAG

Each DAG contributes six unique target nodes:

1. `source_like`
2. `sink_like`
3. `middle`
4. `high_degree`
5. `communication_heavy`
6. `random`

Seven unique execution contexts are generated for each target node:

\[
6\times7=42
\]

If two target roles initially select the same node, unused nodes are chosen as replacements so that all six targets remain unique.

In [ ]:

def choose_six_targets(graph, topo_level, node_df, graph_id):
    rng = np.random.default_rng(stable_seed("targets", graph_id))
    nodes = [str(n) for n in graph.nodes()]
    node_lookup = node_df.set_index("node_id")

    selected = []
    used = set()

    def add(role, node):
        node = str(node)
        if node not in used:
            selected.append((role, node))
            used.add(node)

    sources = [n for n in nodes if graph.in_degree(n) == 0]
    sinks = [n for n in nodes if graph.out_degree(n) == 0]

    if sources:
        add("source_like", max(sources, key=lambda n: graph.out_degree(n)))
    if sinks:
        add("sink_like", max(sinks, key=lambda n: graph.in_degree(n)))

    max_level = max(topo_level.values())
    middle_level = max_level / 2.0
    add(
        "middle",
        min(
            nodes,
            key=lambda n: (
                abs(topo_level[n] - middle_level),
                -(graph.in_degree(n) + graph.out_degree(n)),
            ),
        ),
    )

    add(
        "high_degree",
        max(
            nodes,
            key=lambda n: graph.in_degree(n) + graph.out_degree(n),
        ),
    )

    add(
        "communication_heavy",
        max(
            nodes,
            key=lambda n: int(node_lookup.loc[str(n), "input_size_bytes"]),
        ),
    )

    remaining = [n for n in nodes if n not in used]
    if remaining:
        add("random", str(rng.choice(remaining)))

    fill_index = 1
    while len(selected) < 6:
        remaining = [n for n in nodes if n not in used]
        if not remaining:
            raise RuntimeError(f"Not enough unique targets in graph {graph_id}")
        add(f"diverse_fill_{fill_index}", str(rng.choice(remaining)))
        fill_index += 1

    assert len(selected) == 6
    return selected


## 8) Execution-Time Simulator: Equations (39) and (40)

The execution-time target is generated using the structure of project equation (39), while effective bandwidth follows equation (40).

The project names the cache, memory, and OS interference terms but does not prescribe exact numeric functions for them. Here they are implemented as simple, positive, reproducible functions of the current system state. Their coefficients are explicit simulator choices and are stored in the generation configuration.

The random term \(\epsilon\) is mildly heteroscedastic: unexplained runtime variability increases slightly under higher system load.

In [ ]:

def stratified_beta_load_scores(rng, n=7):
    pool_size = 7000
    pool = np.sort(
        rng.beta(
            LOAD_BETA_A,
            LOAD_BETA_B,
            size=pool_size,
        )
    )

    quantiles = (
        np.arange(n)
        + rng.uniform(0.10, 0.90, size=n)
    ) / n

    indices = np.clip(
        (quantiles * (pool_size - 1)).astype(int),
        0,
        pool_size - 1,
    )

    scores = pool[indices].astype(float)
    rng.shuffle(scores)
    return scores


def sample_system_state(rng, load_score):
    L = float(np.clip(load_score, 0.0, 1.0))

    cpu_utilization = float(np.clip(
        0.14 + 0.78 * L
        + rng.normal(0.0, SYSTEM_STATE_CONFIG["cpu_noise_std"]),
        0.03,
        0.98,
    ))

    bus_utilization = float(np.clip(
        0.10 + 0.76 * L
        + rng.normal(0.0, SYSTEM_STATE_CONFIG["bus_noise_std"]),
        0.02,
        0.98,
    ))

    thermal_pressure = float(np.clip(
        0.04 + 0.72 * L
        + rng.normal(0.0, SYSTEM_STATE_CONFIG["thermal_noise_std"]),
        0.00,
        0.98,
    ))

    ready_queue_length = int(np.clip(
        rng.poisson(0.35 + 5.2 * L),
        0,
        10,
    ))

    active_core_count = int(np.clip(
        round(
            1.0 + 3.0 * L
            + rng.normal(
                0.0,
                SYSTEM_STATE_CONFIG["active_core_noise_std"],
            )
        ),
        1,
        4,
    ))

    memory_active_tasks = int(np.clip(
        round(
            0.15 + 3.6 * L
            + rng.normal(
                0.0,
                SYSTEM_STATE_CONFIG["memory_task_noise_std"],
            )
        ),
        0,
        4,
    ))

    release_jitter_us = float(np.clip(
        rng.gamma(
            shape=2.0,
            scale=12.0 + 55.0 * L,
        ),
        0.0,
        450.0,
    ))

    return {
        "load_score": L,
        "cpu_utilization": cpu_utilization,
        "ready_queue_length": ready_queue_length,
        "active_core_count": active_core_count,
        "memory_active_tasks": memory_active_tasks,
        "bus_utilization": bus_utilization,
        "thermal_pressure": thermal_pressure,
        "release_jitter_us": release_jitter_us,
    }


def hardware_config(core_type: str, dvfs_level: int):
    row = hardware_df[
        (hardware_df["core_type"] == core_type)
        & (hardware_df["dvfs_level"] == dvfs_level)
    ].iloc[0]
    return row.to_dict()


def simulate_execution_time(
    node_row,
    lambda_v,
    core_type,
    dvfs_level,
    state,
    noise_rng,
):
    hw = hardware_config(core_type, dvfs_level)

    effective_ips = float(hw["effective_ips_per_sec"])
    bmax = float(hw["nominal_bandwidth_bytes_per_sec"])

    bw_eff = bmax / (
        1.0
        + ETA_1 * state["memory_active_tasks"]
        + ETA_2 * state["bus_utilization"]
        + ETA_3 * state["thermal_pressure"]
    )

    t_compute = float(node_row["compute_cycles"]) / effective_ips
    t_memory = float(lambda_v) * float(node_row["memory_bytes"]) / bw_eff
    t_communication = (
        GAMMA
        * float(node_row["input_size_bytes"])
        / bw_eff
    )

    memory_pressure = min(
        1.0,
        state["memory_active_tasks"] / 4.0,
    )
    queue_pressure = min(
        1.0,
        state["ready_queue_length"] / 10.0,
    )

    i_cache = (
        CACHE_INTERFERENCE_COEF
        * t_compute
        * (
            0.50 * state["cpu_utilization"]
            + 0.50 * memory_pressure
        )
    )

    i_mem = (
        MEM_INTERFERENCE_COEF
        * t_memory
        * state["bus_utilization"]
        * (1.0 + memory_pressure)
    )

    i_os = (
        OS_INTERFERENCE_COEF
        * t_compute
        * (
            0.50 * state["cpu_utilization"]
            + 0.50 * queue_pressure
        )
        + OS_FIXED_OVERHEAD_US * 1e-6
    )

    deterministic_seconds = (
        t_compute
        + t_memory
        + t_communication
        + i_cache
        + i_mem
        + i_os
    )

    noise_ratio = NOISE_STD_RATIO * (
        0.75 + 0.50 * state["load_score"]
    )

    epsilon = float(
        noise_rng.normal(
            loc=0.0,
            scale=noise_ratio * deterministic_seconds,
        )
    )

    y_exec_seconds = max(
        1e-9,
        deterministic_seconds + epsilon,
    )

    return {
        "frequency_ghz": float(hw["frequency_ghz"]),
        "voltage_v": float(hw["voltage_v"]),
        "effective_ips_per_sec": effective_ips,
        "effective_bandwidth_bytes_per_sec": bw_eff,
        "term_compute_us": t_compute * 1e6,
        "term_memory_us": t_memory * 1e6,
        "term_communication_us": t_communication * 1e6,
        "I_cache_us": i_cache * 1e6,
        "I_mem_us": i_mem * 1e6,
        "I_os_us": i_os * 1e6,
        "epsilon_us": epsilon * 1e6,
        "y_exec_us": y_exec_seconds * 1e6,
    }


In [ ]:

HARDWARE_PAIRS = [
    ("big", 0),
    ("little", 0),
    ("big", 1),
    ("little", 1),
    ("big", 2),
    ("little", 2),
    ("big", 3),
    ("little", 3),
]


def generate_balanced_records(
    graph,
    topo_level,
    node_df,
    simulator_df,
    graph_id,
    split,
):
    rng = np.random.default_rng(stable_seed("records", graph_id))

    targets = choose_six_targets(
        graph,
        topo_level,
        node_df,
        graph_id,
    )

    node_lookup = node_df.set_index("node_id")
    lambda_lookup = simulator_df.set_index("node_id")["lambda_v"]

    rows = []

    graph_pair_offset = int(
        stable_seed("hardware_offset", graph_id)
        % len(HARDWARE_PAIRS)
    )

    for target_index, (target_role, target_node) in enumerate(targets):
        target_node = str(target_node)

        load_scores = stratified_beta_load_scores(
            rng,
            n=7,
        )

        offset = (
            graph_pair_offset + target_index
        ) % len(HARDWARE_PAIRS)

        target_pairs = [
            HARDWARE_PAIRS[
                (offset + context_index)
                % len(HARDWARE_PAIRS)
            ]
            for context_index in range(7)
        ]

        rng.shuffle(load_scores)

        for context_index, ((core_type, dvfs_level), load_score) in enumerate(
            zip(target_pairs, load_scores)
        ):
            core_id = str(rng.choice(CORES[core_type]))

            state = sample_system_state(
                rng,
                float(load_score),
            )

            context_id = (
                f"{graph_id}_t{target_index}_c{context_index}"
            )

            noise_rng = np.random.default_rng(
                stable_seed(
                    "noise",
                    graph_id,
                    target_node,
                    context_id,
                )
            )

            components = simulate_execution_time(
                node_row=node_lookup.loc[target_node],
                lambda_v=float(lambda_lookup.loc[target_node]),
                core_type=core_type,
                dvfs_level=int(dvfs_level),
                state=state,
                noise_rng=noise_rng,
            )

            rows.append({
                "sample_id": f"{split}_{graph_id}_{len(rows):02d}",
                "split": split,
                "graph_id": graph_id,
                "target_node": target_node,
                "target_role": target_role,
                "target_index": target_index,
                "context_index": context_index,
                "context_id": context_id,

                "core_type": core_type,
                "core_id": core_id,
                "dvfs_level": int(dvfs_level),
                "frequency_ghz": components["frequency_ghz"],
                "voltage_v": components["voltage_v"],

                "cpu_utilization": state["cpu_utilization"],
                "ready_queue_length": state["ready_queue_length"],
                "active_core_count": state["active_core_count"],
                "memory_active_tasks": state["memory_active_tasks"],
                "bus_utilization": state["bus_utilization"],
                "thermal_pressure": state["thermal_pressure"],
                "release_jitter_us": state["release_jitter_us"],

                "generator_load_score": state["load_score"],

                "audit_effective_ips_per_sec": components["effective_ips_per_sec"],
                "audit_effective_bandwidth_bytes_per_sec": components["effective_bandwidth_bytes_per_sec"],
                "audit_term_compute_us": components["term_compute_us"],
                "audit_term_memory_us": components["term_memory_us"],
                "audit_term_communication_us": components["term_communication_us"],
                "audit_I_cache_us": components["I_cache_us"],
                "audit_I_mem_us": components["I_mem_us"],
                "audit_I_os_us": components["I_os_us"],
                "audit_epsilon_us": components["epsilon_us"],

                "y_exec_us": components["y_exec_us"],
            })

    assert len(rows) == RECORDS_PER_GRAPH
    assert len({row["context_id"] for row in rows}) == RECORDS_PER_GRAPH

    return pd.DataFrame(rows)


## 9) Dry Run on One DAG

Before generating the complete dataset, this section checks one DAG and displays:

- node features,
- edge features,
- simulator-only parameters,
- task metadata,
- all 42 unique execution contexts,
- the six selected target nodes,
- big/little core balance,
- DVFS coverage,
- latent load coverage.

No final dataset files are written by this dry-run section.

In [ ]:

sample_manifest_row = manifest.iloc[0]

(
    sample_graph,
    sample_topo_order,
    sample_topo_level,
    sample_nodes,
    sample_edges,
    sample_simulator,
    sample_task,
    sample_summary,
) = build_graph_static_data(sample_manifest_row)

sample_records = generate_balanced_records(
    sample_graph,
    sample_topo_level,
    sample_nodes,
    sample_simulator,
    str(sample_manifest_row["graph_id"]),
    str(sample_manifest_row["split"]),
)

print("NODE FEATURES — GNN inputs")
display(sample_nodes.head(10))

print("EDGE FEATURES")
display(sample_edges.head(10))

print("SIMULATOR-ONLY NODE PARAMS")
display(sample_simulator.head(10))

print("TASK METADATA")
display(pd.DataFrame([sample_task]))

print("42 UNIQUE CONTEXTS")
display(sample_records.head(42))

print("Targets")
display(
    sample_records[
        ["target_role", "target_node"]
    ].drop_duplicates()
)

print("Core balance")
display(sample_records["core_type"].value_counts())

print("DVFS balance")
display(
    sample_records["dvfs_level"]
    .value_counts()
    .sort_index()
)

print("Load coverage")
display(
    sample_records["generator_load_score"]
    .describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    )
)


## 10) Generate the Full 25.2K Dataset

This section processes all 600 DAGs. Each DAG produces exactly 42 unique execution contexts, and all records from a graph remain inside the graph's original split.

The final record counts are:

\[
16800+2100+2100+2100+2100=25200
\]

The source DAG dataset is not modified. A new output directory is created at `~/dag_runtime_dataset_25k/`.

In [ ]:
if DEST_ROOT.exists():
    if OVERWRITE_DESTINATION:
        shutil.rmtree(DEST_ROOT)
    else:
        raise RuntimeError(f"{DEST_ROOT} already exists. Set OVERWRITE_DESTINATION=True only if you want to replace it.")

for split in SPLITS:
    (DEST_ROOT / "graphs" / split).mkdir(parents=True, exist_ok=True)
    (DEST_ROOT / "features" / "nodes" / split).mkdir(parents=True, exist_ok=True)
    (DEST_ROOT / "features" / "edges" / split).mkdir(parents=True, exist_ok=True)
    (DEST_ROOT / "simulator" / "node_params" / split).mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "records").mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "tasks").mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "hardware").mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "metadata").mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "schema").mkdir(parents=True, exist_ok=True)

records_by_split = {split: [] for split in SPLITS}
task_rows, summary_rows = [], []

for index, row in manifest.iterrows():
    graph_id = str(row["graph_id"])
    split = str(row["split"])
    graph, topo_order, topo_level, node_df, edge_df, simulator_df, task_row, summary_row = build_graph_static_data(row)
    records_df = generate_balanced_records(graph, topo_level, node_df, simulator_df, graph_id, split)

    shutil.copy2(SOURCE_ROOT / str(row["file_path"]), DEST_ROOT / "graphs" / split / f"{graph_id}.dot")
    node_df.to_csv(DEST_ROOT / "features" / "nodes" / split / f"{graph_id}.csv", index=False)
    edge_df.to_csv(DEST_ROOT / "features" / "edges" / split / f"{graph_id}.csv", index=False)
    simulator_df.to_csv(DEST_ROOT / "simulator" / "node_params" / split / f"{graph_id}.csv", index=False)

    records_by_split[split].append(records_df)
    task_rows.append(task_row)
    summary_rows.append(summary_row)

    if (index + 1) % 50 == 0 or index + 1 == len(manifest):
        print(f"Processed {index + 1}/{len(manifest)} graphs")

all_record_frames = []
for split in SPLITS:
    split_records = pd.concat(records_by_split[split], ignore_index=True)
    split_records.to_csv(DEST_ROOT / "records" / f"{split}.csv", index=False)
    all_record_frames.append(split_records)

all_records = pd.concat(all_record_frames, ignore_index=True)
all_records.to_csv(DEST_ROOT / "records" / "all_records.csv", index=False)

task_metadata_df = pd.DataFrame(task_rows)
task_metadata_df.to_csv(DEST_ROOT / "tasks" / "task_metadata.csv", index=False)

graph_summary_df = pd.DataFrame(summary_rows)
graph_summary_df.to_csv(DEST_ROOT / "metadata" / "graph_summary.csv", index=False)

hardware_df.to_csv(DEST_ROOT / "hardware" / "hardware_dvfs.csv", index=False)
shutil.copy2(SOURCE_MANIFEST, DEST_ROOT / "metadata" / "source_all_graphs_manifest.csv")

print("Dataset generated at:", DEST_ROOT)

## 11) Save the Model Schema and Generation Configuration

`model_feature_schema.json` explicitly records which columns are allowed to enter the future model.

`generation_config.json` records the random seed, simulator constants, node-distribution parameters, system-state generation parameters, hardware configuration, and expected split sizes.

The following are generation, simulator, or audit metadata and must not be used as model inputs:

- `generator_load_score`
- `lambda_v`
- `target_role`
- target/context indices and IDs
- `audit_*`
- task period/deadline metadata
- physical `core_id`

In [ ]:
MODEL_FEATURE_SCHEMA = {
    "gnn_node_features": [
        "compute_cycles",
        "memory_bytes",
        "input_size_bytes",
        "in_degree",
        "out_degree",
        "operation_type",
        "criticality",
        "topo_level",
        "reverse_topo_level",
    ],
    "gnn_edge_features": ["data_bytes"],
    "quantile_head_core_features": ["core_type"],
    "quantile_head_dvfs_features": ["frequency_ghz", "voltage_v"],
    "z_t_features": [
        "cpu_utilization",
        "ready_queue_length",
        "active_core_count",
        "memory_active_tasks",
        "bus_utilization",
        "thermal_pressure",
        "release_jitter_us",
    ],
    "target": "y_exec_us",
    "not_model_inputs": [
        "lambda_v",
        "period_us",
        "deadline_us",
        "core_id",
        "generator_load_score",
        "target_role",
        "target_index",
        "context_index",
        "context_id",
        "audit_*",
    ],
}

GENERATION_CONFIG = {
    "base_seed": BASE_SEED,
    "source_root": str(SOURCE_ROOT),
    "destination_root": str(DEST_ROOT),
    "records_per_graph": RECORDS_PER_GRAPH,
    "total_records": sum(EXPECTED_RECORD_COUNTS.values()),
    "split_record_counts": EXPECTED_RECORD_COUNTS,
    "node_profiles": NODE_PROFILES,
    "dvfs_table": DVFS_TABLE,
    "core_capacity_factor": CORE_CAPACITY_FACTOR,
    "cores": CORES,
    "eta_1": ETA_1,
    "eta_2": ETA_2,
    "eta_3": ETA_3,
    "gamma": GAMMA,
    "cache_interference_coef": CACHE_INTERFERENCE_COEF,
    "mem_interference_coef": MEM_INTERFERENCE_COEF,
    "os_interference_coef": OS_INTERFERENCE_COEF,
    "os_fixed_overhead_us": OS_FIXED_OVERHEAD_US,
    "noise_std_ratio": NOISE_STD_RATIO,
    "reference_ips": REFERENCE_IPS,
    "reference_bw": REFERENCE_BW,
    "system_state_config": SYSTEM_STATE_CONFIG,
    "load_beta_a": LOAD_BETA_A,
    "load_beta_b": LOAD_BETA_B,
    "notes": {
        "pdf_fixed_parameters": "DVFS frequency/voltage values and formulas (39),(40).",
        "implementation_chosen_parameters": "Feature ranges, effective IPS factors, nominal bandwidths, eta/interference coefficients, noise scale and continuous system-state generation parameters are explicit simulator choices because the PDF does not assign their numeric values."
    },
}

with (DEST_ROOT / "schema" / "model_feature_schema.json").open("w", encoding="utf-8") as file:
    json.dump(MODEL_FEATURE_SCHEMA, file, indent=2, ensure_ascii=False)
with (DEST_ROOT / "generation_config.json").open("w", encoding="utf-8") as file:
    json.dump(GENERATION_CONFIG, file, indent=2, ensure_ascii=False)

print(json.dumps(MODEL_FEATURE_SCHEMA, indent=2, ensure_ascii=False))

## 12) Strict Validation and Distribution Checks

The final validation verifies all major dataset contracts, including:

- exactly 42 unique contexts per DAG,
- exactly 6 unique target nodes per DAG,
- exactly 21 big-core and 21 little-core records per DAG,
- broad and approximately balanced DVFS coverage,
- valid ranges for every component of \(z_t\),
- the same system-state generator across all splits,
- positive non-missing runtime targets,
- \(D_i\le T_i\),
- expected graph and record counts per split,
- `input_size_bytes` exactly equal to the sum of incoming `data_bytes` values.

The printed split-wise statistics are intended to expose unintended distribution drift before model training.

In [ ]:

record_counts = all_records.groupby("split").size().to_dict()

assert record_counts == EXPECTED_RECORD_COUNTS, record_counts
assert len(all_records) == 25200

assert (
    all_records.groupby("graph_id").size()
    == RECORDS_PER_GRAPH
).all()

assert (
    all_records.groupby(["graph_id", "context_id"]).size()
    == 1
).all()

assert (
    all_records.groupby("graph_id")["target_node"].nunique()
    == 6
).all()

core_counts = (
    all_records
    .groupby(["graph_id", "core_type"])
    .size()
    .unstack(fill_value=0)
)

assert (core_counts["big"] == 21).all()
assert (core_counts["little"] == 21).all()

assert all_records["y_exec_us"].notna().all()
assert (all_records["y_exec_us"] > 0).all()

assert all_records["cpu_utilization"].between(0, 1).all()
assert all_records["bus_utilization"].between(0, 1).all()
assert all_records["thermal_pressure"].between(0, 1).all()
assert all_records["active_core_count"].between(1, 4).all()
assert all_records["memory_active_tasks"].between(0, 4).all()
assert all_records["generator_load_score"].between(0, 1).all()

assert (
    task_metadata_df["deadline_us"]
    <= task_metadata_df["period_us"]
).all()

assert (
    graph_summary_df.groupby("split").size().to_dict()
    == EXPECTED_GRAPH_COUNTS
)

for split in SPLITS:
    expected = EXPECTED_GRAPH_COUNTS[split]

    assert len(list((DEST_ROOT / "graphs" / split).glob("*.dot"))) == expected
    assert len(list((DEST_ROOT / "features" / "nodes" / split).glob("*.csv"))) == expected
    assert len(list((DEST_ROOT / "features" / "edges" / split).glob("*.csv"))) == expected
    assert len(list((DEST_ROOT / "simulator" / "node_params" / split).glob("*.csv"))) == expected

for row in manifest.itertuples(index=False):
    graph_id = str(row.graph_id)
    split = str(row.split)

    node_df = pd.read_csv(
        DEST_ROOT / "features" / "nodes" / split / f"{graph_id}.csv",
        dtype={"node_id": str},
    )

    edge_df = pd.read_csv(
        DEST_ROOT / "features" / "edges" / split / f"{graph_id}.csv",
        dtype={"source": str, "target": str},
    )

    incoming = (
        edge_df.groupby("target")["data_bytes"].sum().to_dict()
        if len(edge_df)
        else {}
    )

    for node_row in node_df.itertuples(index=False):
        assert int(node_row.input_size_bytes) == int(
            incoming.get(str(node_row.node_id), 0)
        )

print("All validation checks passed.")
print()
print("Record counts:")
print(record_counts)
print("Total:", len(all_records))

print("\nCore-type counts by split")
display(
    pd.crosstab(
        all_records["split"],
        all_records["core_type"],
    )
)

print("\nDVFS counts by split")
display(
    pd.crosstab(
        all_records["split"],
        all_records["dvfs_level"],
    )
)

print("\nLoad-score distribution")
display(
    all_records
    .groupby("split")["generator_load_score"]
    .describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    )
)

z_columns = [
    "cpu_utilization",
    "ready_queue_length",
    "active_core_count",
    "memory_active_tasks",
    "bus_utilization",
    "thermal_pressure",
    "release_jitter_us",
]

print("\nz_t means by split")
display(
    all_records
    .groupby("split")[z_columns]
    .mean()
)

print("\nz_t std by split")
display(
    all_records
    .groupby("split")[z_columns]
    .std()
)

print("\ny_exec_us summary")
display(
    all_records
    .groupby("split")["y_exec_us"]
    .agg(["min", "median", "mean", "std", "max"])
)


# Dataset Ready for GNN Training

After the validation cell completes successfully, the final dataset is available at:

```text
~/dag_runtime_dataset_25k/
```

Expected record counts:

```text
train        16800
validation    2100
calibration   2100
test_id       2100
test_ood      2100
------------------
total        25200
```

Each DAG has exactly 42 unique execution contexts.

The future model should use:

- node features from `features/nodes/...`,
- graph topology from the DAG edges,
- edge attribute `data_bytes`,
- core/DVFS/system-state context from `records/...`,
- target `y_exec_us`.

The following must **not** be used as model inputs:

- `lambda_v`
- `generator_load_score`
- `audit_*`
- `deadline_us`
- `period_us`
- `core_id`
- target-selection metadata
- context identifiers

Feature normalization must be fitted using the **training split only**, then applied unchanged to validation, calibration, test-ID, and test-OOD.